# Rust 123: Macros — Example

The closing chapter of the course: the machinery behind `println!`,
`assert_eq!` and `#[test]` itself. Declarative macros (`macro_rules!`) are
pattern matching over **tokens** — the same match-and-rebind idea as §3 of
chapter 03, but at compile time. Twin: `26_macros_p.ipynb`. Python analogies
(decorators, `__getattr__`) appear where they genuinely help.


### Contents

- **0.** Conventions: kernel rules, the desk (`0.1`, `0.2`)
- **1.** First macros — shape matching (`1.1`), repetition (`1.2`), `stringify!` (`1.3`)
- **2.** Fragment specifiers — the running desk build (`2.1`), `expr`/`ident` (`2.2`), unmatched input (`2.3`), unused/shadowed macros (`2.4`)
- **3.** Multi-arm matching — routing (`3.1`), the catch-all arm (`3.2`)
- **5.** Hygiene — the inner name stays in (`5.1`), passing names through `ident` (`5.2`)
- **6.** Textual scope — definition before call (`6.1`)
- **7.** Traps — positional args and the diagnostic table (`7.1`)
- **8.** The Python mapping
- **9.** Cheat sheet
- **10.** Review
- **99.** Desk teardown


## 0. Notebook Conventions

- The kernel strips `#[test]` fns and traits typed at cell top level (ch17).
  Bigger rule for macros: **`macro_rules!` has *textual* scope** — a macro is
  visible from its definition *downward* in the file. In the kernel the
  definition persists like any item, so define **before** the cells that
  call it; the gate enforces the same order.
- Persisted variables must be owned (`'static`): no top-level `&str` slices,
  no `Vec<&str>`; use `String`, `Vec<String>`, `HashMap<String, _>` or wrap
  in `{}` (ch25 §17).
- Statements in cells type `let`, `for`, `if`, `match` internally; the prelude
  has `println!`, `vec!`, `assert!`/`assert_eq!` — all macros themselves.
- The desk: `rustc` compiles `scratch26/desk_<name>.rs`; helper `desk` writes
  a source and invokes it with args, returning `(stdout, code)`. Windows AV
  may veto a *first* run from a fresh `.exe`; rerun.
- `echo_rs` is the running desk build (§1): edit it in place (or write your
  own `scratch26/echo.rs`) and rerun to interact with macros at compile time.


In [ ]:
// 0.1 Smoke: the toolchain. Macros are compile-time, so the version line is
// the ground truth the desk reports against.
use std::process::Command;

let v = Command::new("rustc")
    .arg("--version")
    .output()
    .unwrap();
let version = String::from_utf8_lossy(&v.stdout).trim().to_string();
println!("toolchain: {version}");
assert!(version.starts_with("rustc"), "got: {version}");


### 0.2 — The desk

Two helpers, ~30 lines, owned once: `desk` (args in, `(stdout, code)` out)
and `desk_build` (a two-line convenience that compiles and returns the
compiler's stderr — the §2.4/§5 programme reads *that* as data).


In [ ]:
// 0.2 The desk. `desk(name, args)` writes nothing: it runs the already-built
// scratch26/desk_<name>.exe. `desk_build(name, src)` compiles scratch source
// and returns the compiler's stderr ("" when clean).
use std::fs;

/// Run the compiled desk program `name` with `args`; return (stdout, code).
fn desk(name: &str, args: &[&str]) -> (String, i32) {
    let run = Command::new(format!("./scratch26/desk_{name}.exe"))
        .args(args)
        .output()
        .unwrap();
    let out = String::from_utf8_lossy(&run.stdout).into_owned();
    let code = run.status.code().unwrap_or(-1);
    (out, code)
}

/// Write `src` to scratch26/desk_<name>.rs and compile it; return stderr.
/// Empty string = clean build. Errors and warnings are both data here.
fn desk_stderr(name: &str, src: &str) -> String {
    fs::create_dir_all("scratch26").unwrap();
    let rs = format!("scratch26/desk_{name}.rs");
    let exe = format!("scratch26/desk_{name}.exe");
    fs::write(&rs, src).unwrap();
    let out = Command::new("rustc")
        .args(["--edition", "2024", "-o", &exe, &rs])
        .output()
        .unwrap();
    String::from_utf8_lossy(&out.stderr).into_owned()
}

/// Compile `src` and run it with `args`: the usual two-step.
/// Panics on any compiler stderr -- watch-desk sources with warnings are for
/// `desk_stderr`, not this path.
fn desk_run(name: &str, src: &str, args: &[&str]) -> (String, i32) {
    let stderr = desk_stderr(name, src);
    assert!(
        stderr.is_empty(),
        "desk_{name} failed to compile:\n{stderr}"
    );
    desk(name, args)
}

println!("desk helpers ready (desk, desk_stderr, desk_run)");


## 1. First Macros

A `macro_rules!` definition is a name plus **arms**; an arm is `matcher =>
transcriber` — tokens in, tokens out. The compiler rewrites the call site
*before* name resolution, so a macro can produce identifiers, not just
values. There is no Python analog: this is parse-time metaprogramming.


In [ ]:
// 1.1 Two macros that match by shape -- same fragment type, different
// delimiters. Both expand to expressions.
macro_rules! add {
    ($a:expr, $b:expr) => {
        $a + $b
    };
}

macro_rules! twice {
    [$x:expr] => {
        $x * 2
    };
}

let sum = add!(19, 23);
let dbl = twice![21];
println!("add!(19, 23) = {sum}, twice![21] = {dbl}");
assert_eq!(sum, 42);
assert_eq!(dbl, 42);
println!("1.1 receipt: two shapes, one expansion rule each -- both 42");


### 1.2 — Repetition

`$($x:expr),*` is the macro engine's `for` loop: zero or more comma-separated
`expr` fragments, each expanded once per `$x` occurrence in the transcriber.
The optional `$(,)?` tail swallows a trailing comma. Block-bodied arms
(`{{ ... }}`) build statements — Python's closest sight is a comprehension
that grew statements.


In [ ]:
// 1.2 Repetition: zero or more, comma-separated, statement-building.
macro_rules! tally {
    ($($n:expr),*) => {{
        #[allow(unused_mut)] // the empty call never mutates
        let mut total = 0;
        $( total += $n; )*
        total
    }};
}

let t = tally![1, 2, 3, 4];
println!("tally![1, 2, 3, 4] = {t}");
assert_eq!(t, 10);
let empty = tally![];
println!("tally![] = {empty}");
assert_eq!(empty, 0);
println!("1.2 receipt: zero or more -- the * of macro land");


### 1.3 — `stringify!`: the receipt is spelling

`stringify!` yields its input's **token spelling** as a `&str`. Single tokens
get single spaces around them; your original grouping survives otherwise.


In [ ]:
// 1.3 Spelling as data: what the compiler actually sees.
let spelled = stringify!(1 + 2 * 3);
let call = stringify!(add!(19, 23));
println!("stringify!(1 + 2 * 3) = {spelled:?}");
println!("stringify!(add!(19, 23)) = {call:?}");
assert_eq!(spelled, "1 + 2 * 3");
assert_eq!(call, "add!(19, 23)");
println!("1.3 receipt: tokens — name! binds tight, ops get spaces");


## 2. Fragment Specifiers

The part after the colon decides how the compiler treats matched tokens:
`expr` parses an expression tree; `ident` matches one identifier; bare
literals (like `mature`) match themselves. Matching is by **shape**, not by
text.


### 2.1 — The running desk build

`ECHO_RS` below is a complete `rustc` program with one `macro_rules!` inside:
`special_returns!` maps the literal token `mature` to `1.0` and *any other
single token tree* (`:tt`) to `0.0`. The build is **given**; §2.1 compiles and
runs it. From then on it is yours: edit
[scratch26/echo_bench.rs](scratch26/echo_bench.rs) in place, rerun the cell —
each run recompiles from the file on disk.


In [ ]:
// 2.1 Given: the running desk build's source. Compile it, run it both ways.
// (fs is already imported in 0.2 -- one-namespace rule across cells)

const ECHO_RS: &str = r#"macro_rules! special_returns {
    (mature) => {
        1.0_f64
    };
    ($other:tt) => {
        0.0_f64
    };
}

fn main() {
    let arg: Vec<String> = std::env::args().skip(1).collect();
    let cmd = arg.first().map(String::as_str).unwrap_or("");
    match cmd {
        "special_returns" => println!("{}", special_returns!(mature)),
        "any_token" => println!("{}", special_returns!(whatever)),
        _ => println!("unknown command"),
    }
}
"#;

fs::create_dir_all("scratch26").unwrap();
fs::write("scratch26/echo_bench.rs", ECHO_RS).unwrap();
let (mature, code1) = desk_run("echo_bench", ECHO_RS, &["special_returns"]);
let (other, code2) = desk("echo_bench", &["any_token"]);
println!("special_returns!(mature)   = {}", mature.trim());
println!("special_returns!(whatever) = {}", other.trim());
assert_eq!((mature.trim(), other.trim()), ("1", "0"));
assert_eq!(code1 + code2, 0);
println!("2.1 receipt: the literal arm wins; :tt catches the rest");


### 2.2 — `expr` vs `ident`

`ident` fragments become real bindings: the caller's name, declared by the
macro. Together with `=>` separators and a base-plus-repetition tally, this
cell is the everyday toolkit.


In [ ]:
// 2.2 ident = real bindings; => builds pairs; base + repetition tallies.
macro_rules! binding {
    ($name:ident = $val:expr) => {
        let $name = $val;
    };
}

macro_rules! kv_map {
    ($($k:expr => $v:expr),* $(,)?) => {{
        let mut m = std::collections::HashMap::new();
        $( m.insert($k, $v); )*
        m
    }};
}

macro_rules! sum {
    ($base:expr $(, $rest:expr)*) => {{
        let mut acc = $base;
        $( acc += $rest; )*
        acc
    }};
}

let mut x = 0;
binding!(qty = 5);
let _ = &mut x;
println!("binding!: qty={qty}");
assert_eq!(qty, 5);
let m = kv_map!["a" => 1, "b" => 2];
println!("kv_map![..] len = {}", m.len());
assert_eq!(m.len(), 2);
let m2 = kv_map!["c" => 3,];
println!("trailing comma accepted, len = {}", m2.len());
assert_eq!(m2.len(), 1);
let total = sum![10, 20, 30];
println!("sum![10, 20, 30] = {total}");
assert_eq!(total, 60);
println!("2.2 receipt: ident makes bindings, => builds pairs");


### 2.3 — Unmatched input is a compile error

No arm matches `1, 2, 3` for a `(expr)` matcher — the compiler stops at
the call site with `no rules expected this token in macro call`, naming the
meta-variable it was trying to match. Desk time: the error text is the receipt.


In [ ]:
// 2.3 The compile error, captured. Desk source, desk_stderr, data.
const ARM_SRC: &str = r#"macro_rules! double {
    ($x:expr) => {
        $x * 2
    };
}

fn main() {
    let _ = double!(1, 2, 3);
}
"#;

let stderr = desk_stderr("noarm", ARM_SRC);
println!("--- compile stderr:\n{stderr}");
assert!(stderr.contains("no rules expected this token in macro call"), "stderr: {stderr}");
assert!(stderr.contains("while trying to match meta-variable `$x:expr`"), "stderr: {stderr}");
println!("2.3 receipt: arm mismatch is a call-site compile error");


### 2.4 — The unused macro

A defined-but-never-invoked macro earns `unused macro definition` — the
`#[warn(unused_macros)]` family. Redefining the same name is *legal*: the
second definition shadows the first, calls resolve to the newcomer, and the
original copy becomes an unused-macro warning pointing at *its* line.


In [ ]:
// 2.4 Both silences, desk-verified.
const NEVER_SRC: &str = r#"macro_rules! never_invoked {
    () => {
        1
    };
}

fn main() {}
"#;

let stderr = desk_stderr("neverused", NEVER_SRC);
println!("--- unused stderr:\n{stderr}");
assert!(stderr.contains("warning: unused macro definition: `never_invoked`"), "stderr: {stderr}");

const TWICE_SRC: &str = r#"macro_rules! echo_twice {
    ($x:expr) => {
        $x
    };
}

macro_rules! echo_twice {
    ($x:expr) => {
        $x
    };
}

fn main() {
    let _ = echo_twice!(7);
}
"#;

let stderr = desk_stderr("twicewarn", TWICE_SRC);
println!("--- twice stderr:\n{stderr}");
assert!(stderr.contains("warning: unused macro definition: `echo_twice`"), "stderr: {stderr}");
assert!(!stderr.contains("error"), "stderr: {stderr}");
println!("2.4 receipt: unused_macros warns; redefinition shadows (first copy warned)");


## 3. Multi-Arm Matching

Arms are tried top to bottom; the first matcher that accepts the input wins.
Arms differ by separators and token trees — shapes, not regexes. A trailing
`$(,)?` on every arm keeps call sites flexible.


In [ ]:
// 3.1 A shape router: three arms, tried in order. Plus a macro-generated
// match with a macro-built arm list -- plus the `_` arm every real match needs.
macro_rules! route {
    (buy $qty:expr) => {
        format!("BUY x{}", $qty)
    };
    (sell $qty:expr) => {
        format!("SELL x{}", $qty)
    };
    (hold) => {
        "HOLD".to_string()
    };
}

macro_rules! rank {
    ($($name:ident => $n:expr),* $(,)?) => {
        match FIRST {
            $( $name => $n, )*
            _ => 0,
        }
    };
}

let a = route!(buy 100);
let b = route!(sell 40);
let c = route!(hold);
println!("{a} | {b} | {c}");
assert_eq!((a.as_str(), b.as_str(), c.as_str()), ("BUY x100", "SELL x40", "HOLD"));

const A: i32 = 7;
const B: i32 = 8;
const FIRST: i32 = A;
let r = rank![A => 1, B => 2];
println!("rank![A => 1, B => 2] = {r}");
assert_eq!(r, 1);
println!("3.1 receipt: literal arms route; repetition builds match arms (+ `_`)");


### 3.2 — Arm order is semantics

With `($($t:tt)*)` as the second arm, nothing falls through — `:tt` eats any
single token tree. A catch-all arm makes earlier arms *dead* if listed after
it; the compiler is silent about it.


In [ ]:
// 3.2 The catch-all swallows; order matters but rustc does not police it.
macro_rules! verdict {
    (ok $n:expr) => {
        format!("ok: {n}", n = $n)
    };
    ($($t:tt)*) => {
        "rejected".to_string()
    };
}

let a = verdict!(ok 5);
let b = verdict!(whatever 1, 2);
println!("{a:?} | {b:?}");
assert_eq!(a, "ok: 5");
assert_eq!(b, "rejected");
println!("3.2 receipt: catch-all last; nothing after it is ever matched");


## 5. Macro Hygiene

`macro_rules!` is hygienic: identifiers created *inside* a macro stay inside.
The classic counter-example: a macro whose body is `let a = 42;` does **not**
introduce an `a` the caller can see — the caller's `let a = 9;` wins.


In [ ]:
// 5.1 The hygiene receipt, kernel-side: the macro's a is invisible.
macro_rules! using_a {
    () => {
        #[allow(unused_variables)] // the whole point: this a dies
        let a = 42;
    };
}

let a = 9;
using_a!();
println!("outer a = {a}");
assert_eq!(a, 9);
println!("5.1 receipt: inner a stayed inside (rustc would warn; allowed here)");


### 5.2 — Passing names through `ident`

Hygiene is not a wall: fragments captured in the matcher are *caller-site*
identifiers. `$name:ident` in, `let $name` out — the binding lands in the
caller's scope (2.2's `binding!`, now as a desk receipt too).


In [ ]:
// 5.2 ident escapes the wall by design: the caller's name, the caller's scope.
macro_rules! declare {
    ($name:ident = $val:expr) => {
        let $name = $val;
    };
}

declare!(price = 228.60);
println!("price = {price}");
assert_eq!(price, 228.60);

// Hygiene + ident in one desk receipt: the macro's own a vs the passed one.
const HYGIENE_SRC: &str = r#"macro_rules! using_a {
    () => {
        let a = 42;
    };
}

macro_rules! pass_a {
    ($a:ident) => {
        let $a = 42;
    };
}

fn main() {
    let a = 9;
    using_a!();
    pass_a!(b);
    println!("{a} {b}");
}
"#;

// desk_stderr only: this source compiles WITH a warning (the macro's unused a)
let stderr = desk_stderr("hyg", HYGIENE_SRC);
let (out, code) = desk("hyg", &[]);
println!("run output: {out:?}");
assert!(out.trim() == "9 42", "out: {out:?}");
assert!(stderr.contains("unused variable: `a`"), "stderr: {stderr}");
assert_eq!(code, 0);
println!("5.2 receipt: inner a dies, passed b lives");


## 6. Textual Scope: Order Is Everything

Functions are order-independent; macros are not. `macro_rules!` lives in
**textual order**: a macro is visible from its definition *downward* only —
use-before-define is `cannot find macro`, even inside the same file.


In [ ]:
// 6.1 Desk receipt: call above definition = cannot find macro.
const ORDER_SRC: &str = r#"fn main() {
    println!("{}", double!(3));
}

macro_rules! double {
    ($x:expr) => {
        $x * 2
    };
}
"#;

let stderr = desk_stderr("order", ORDER_SRC);
println!("--- order stderr:\n{stderr}");
assert!(stderr.contains("cannot find macro `double` in this scope"), "stderr: {stderr}");
assert!(stderr.contains("consider moving the definition of `double` before this call"), "stderr: {stderr}");
println!("6.1 receipt: definitions flow downward only");


### 6.2 — The kernel agrees

In evcxr the definition persists like any item, so the ordering rule is
between *cells*: define in one cell, use in later cells (1.1 → 3.1 already
demonstrated it). There is no kernel way to "use before define" — the gate
mirrors this by hoisting macros in **cell order**, first definition wins.


## 7. Traps: What a Matcher Refuses

Positional arguments are a Python habit, not a Rust macro feature. `$1:expr`
is a syntax error at *definition* time — metavariables are identifiers.


In [ ]:
// 7.1 Definition-time rejection of positional args.
const POS_SRC: &str = r#"macro_rules! pos {
    ($1:expr) => {
        $1
    };
}

fn main() {}
"#;

let stderr = desk_stderr("pos", POS_SRC);
println!("--- pos stderr:\n{stderr}");
assert!(stderr.contains("expected identifier, found `1`"), "stderr: {stderr}");
assert!(stderr.contains("macro_rules! pos"), "stderr: {stderr}");
println!("7.1 receipt: metavariables are named, never numbered");


| Diagnostic | The story it tells |
|---|---|
| `no rules expected this token in macro call` | no arm matched the input's shape (§2.3) — the note names the meta-variable being matched |
| `cannot find macro ... in this scope` | definition is below the call (§6.1) — macros read downward, move it up |
| `missing fragment specifier` | stray `=>`/token inside a matcher (probe-verified) |
| `expected identifier, found \`1\`` | positional metavariable `$1` (§7.1) |
| `unused macro definition` | defined, never invoked — or shadowed by a redefinition (§2.4) |
| `warning: unused variable` from hygiene | the macro's inner binding died (§5.1) |


## 8. The Python Mapping

| Rust macro | Closest Python thing | The difference that matters |
|---|---|---|
| `macro_rules!` arm | no analog (closest: `match` in `__getattr__` dispatch) | matching happens on **tokens at compile time** |
| repetition `$($x:expr),*` | `*args` / `*elements` unpacking | expands *code*, not values |
| `binding!` via `$name:ident` | `locals()[name] = val` / `globals()` | hygienic: real bindings, compile-time checked |
| `stringify!` | `inspect`/`ast.dump` | token spelling, zero runtime cost |
| hygiene | closures' local scope | identifier identity is compile-time, call-site-stamped |
| textual order | module top-to-bottom execution | definitions must precede calls *textually* |
| `#[test]` (from ch25) | `@unittest` decorators | an attribute macro — §WCN below |


### What Comes Next

That is the course. `assert_eq!` was a declarative macro; `#[test]` an
attribute one; the harness you drove all chapter was a macro system matching
tokens and stamping out code. Everything from chapter 01's `println!` to
chapter 25's receipts came through the machinery you now write yourself.


## 9. Cheat Sheet

```rust
// definition: name, arms, matcher => transcriber
macro_rules! name {
    (lit $x:expr)        => { /* tokens out */ };
    ($($k:expr => $v:expr),* $(,)?) => {{ /* statement block */ }};
    (base $b:expr $(, $r:expr)*) => {{ /* base + repetition */ }};
}

// fragments: expr ident tt literal-tokens; separators: , => ; delimiters () [] {}
// repetition: $( ... ) sep*  /  $( ... )?  — zero or more / optional
// hygiene: inner idents stay inner; ident fragments are caller-site names
// order: definitions are visible downward only (textual scope)
// debugging: stringify! for spelling; rustc --pretty or cargo expand for full expansion
```


## 10. Review

**Q: Why did §6.1's desk source fail to compile when the macro was defined
below `main`?** Rust expands macros *textually*, top to bottom: at the call
site the definition has not been seen yet. Functions hoist; macros do not.

**Q: How is `binding!` able to introduce a new variable in my function?**
The `$name:ident` fragment is captured at the call site and re-emitted into a
`let` — hygiene marks it as *yours*. Bindings made without a fragment stay
inside the macro (§5.1).

**Q: What is the difference between `$x:expr` and `$x:tt`?** `expr` parses a
full expression tree; `tt` takes exactly one token tree (a single token or a
balanced `(..)`/`[..]`/`{..}` group). The catch-all arm `($($t:tt)*)` matches
*anything* — which is why arm order matters (§3.2).

**Q: How do I debug a macro that is not expanding the way I expect?**
`stringify!` shows the token spelling of a call (§1.3). For the expansion of
*your* macro, write the call into the desk, compile, and read the compiler's
`usage:` note or expansion errors back as data.

**Q: Where does `#[test]` fit in?** It is an attribute **macro** — not
`macro_rules!` but proc-macro land (rustc's built-in `test` macro). Same
token-in/code-out idea; you will meet `macro_rules!` cousins like `lazy_static!`
and `builder` derive crates in the wild.


In [ ]:
// 99 Desk teardown: scratch26 out.
use std::path::Path;

let existed = Path::new("scratch26").exists();
if existed {
    fs::remove_dir_all("scratch26").unwrap();
}
println!("scratch26 existed: {existed} -- removed: {existed}");
